In [32]:
!git clone https://github.com/laizesuelia/IC-Controle-inteligente-para-PA.git

fatal: destination path 'IC-Controle-inteligente-para-PA' already exists and is not an empty directory.


In [33]:
%matplotlib inline


In [34]:
import sys
sys.path.append('/content/IC-Controle-inteligente-para-PA')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from simulacao import run_simulacao

# ========================================================
# 1. CONFIGURAÇÃO DO MONTE CARLO E TABELA DE MÉTRICAS
# ========================================================
N_simulacoes = 100
passos_totais = 500
MAP_ref = 100.0
modelos = ['GPC_Fixo', 'GPC_Adaptativo', 'Controlador_Perturbacao', 'Controlador_Restricao', 'IDC']

acumulo_V = {m: [] for m in modelos}
acumulo_sigma = {m: [] for m in modelos}
acumulo_pb1 = {m: [] for m in modelos}

print(f"Iniciando Simulações de Monte Carlo ({N_simulacoes} rodadas)...")
print("Isso pode levar alguns segundos dependendo do processador.\n")

for i in range(N_simulacoes):
    res = run_simulacao(passos_totais=passos_totais, sigma_e=1.5, semente=i)
    for m in modelos:
        map_array = np.array(res[m]['historico_MAP'][40:])
        erro = map_array - MAP_ref
        
        V_bar = np.mean(erro**2)
        acumulo_V[m].append(V_bar)
        
        sigma_v = np.std(erro)
        acumulo_sigma[m].append(sigma_v)
        
        pb1_array = np.array(res[m]['hist_var_b1'][40:])
        pb1_bar = np.mean(pb1_array)
        acumulo_pb1[m].append(pb1_bar)

print("Monte Carlo concluído! Calculando médias estocásticas...\n")
print(f"{'Controlador':<25} | {'Perda Regulação (V_bar)':<25} | {'Desvio Padrão (s_v)':<20} | {'Var. Média b1 (p_b1)':<20}")
print("-" * 100)
for m in modelos:
    media_V = np.mean(acumulo_V[m])
    media_sigma = np.mean(acumulo_sigma[m])
    media_pb1 = np.mean(acumulo_pb1[m])
    print(f"{m:<25} | {media_V:<25.4f} | {media_sigma:<20.4f} | {media_pb1:<20.4f}")

# ========================================================
# 2. GERAÇÃO DOS GRÁFICOS MATPLOTLIB
# ========================================================
resultados = run_simulacao(passos_totais=500, sigma_e=0.5)
modelo_escolhido = 'IDC'
dados = resultados[modelo_escolhido]

historico_MAP = dados['historico_MAP']
historico_I = dados['historico_I']
hist_a1 = dados['hist_a1']
real_a1 = dados['real_a1']
hist_b1 = dados['hist_b1']
real_b1 = dados['real_b1']
hist_bm1 = dados['hist_bm1']
real_bm1 = dados['real_bm1']
d0_hist = dados['d0_hist']
d0_real = dados['d0_real']
hist_custo = dados['hist_custo']
hist_var_b1 = dados['hist_var_b1']
historico_rho = dados['hist_rho']

chaves_modelos = ['Controlador_Restricao', 'GPC_Fixo', 'GPC_Adaptativo', 'IDC']
nomes_legenda = ['1. Controlador de Restrição', '2. GPC Clássico', '3. GPC Adaptativo', '4. IDC']
cores = ['tab:red', 'tab:blue', 'tab:orange', 'tab:green']
eventos = [200, 350]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# GRÁFICO 1: Pressão Arterial
ax1.axhline(MAP_ref, color='black', linestyle='--', linewidth=2, label='Referência (100 mmHg)')
for i, chave in enumerate(chaves_modelos):
    ax1.plot(resultados[chave]['historico_MAP'], label=nomes_legenda[i], color=cores[i], alpha=0.8, linewidth=1.5)
for ev in eventos:
    ax1.axvline(ev, color='gray', linestyle=':', alpha=0.7)
ax1.set_title('Comparação de Desempenho: Regulação da Pressão Arterial (MAP)')
ax1.set_ylabel('MAP (mmHg)')
ax1.legend(loc='upper right', ncol=2)
ax1.grid(True, alpha=0.3)

# GRÁFICO 2: Taxa de Infusão
ax2.axhline(180, color='red', linestyle=':', alpha=0.5, label='Saturação Máxima')
for i, chave in enumerate(chaves_modelos):
    ax2.plot(resultados[chave]['historico_I'], label=nomes_legenda[i], color=cores[i], alpha=0.75)
for ev in eventos:
    ax2.axvline(ev, color='gray', linestyle=':', alpha=0.7)
ax2.set_title('Esforço de Controle: Ação da Bomba de Infusão')
ax2.set_xlabel('Passos de Simulação (k)')
ax2.set_ylabel('Infusão (mL/h)')
ax2.legend(loc='upper right', ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()